# SoRLTrainerv8 — Self-Distillation Pipeline Demo

**Goal**: Visualize each stage of the v8 training pipeline:
1. Load model + data, show a sample batch
2. Build compressed sequence: `[query][abs×K][answer]` (CoT removed)
3. Teacher forward → `L_base`, `h_teacher` at `####` position
4. Student forward (recursion iteration) → `L_compress`, `h_student` at `####` position
5. Compute `L_KD = L1(sg(h_teacher), h_student)`
6. Visualize hidden-state alignment across iterations

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from sorl.selfroute import SoRLTrainerv6
from sorl.sorl_trainer import get_answer_start_index
from data.pt_dataset import get_dataset

device = "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen3-0.6B"
ABS_VOCAB = 32
K = 8          # number of abstract prefix tokens
ANSWER_TOK = 820  # #### delimiter

# Load model
model = SorlModelWrapper.from_pretrained(MODEL_NAME, abstract_vocab_size_list=[ABS_VOCAB])
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model = model.to(device).eval()

bv = int(model.vocab_sizes[0].item())
print(f"Model loaded: base_vocab={bv}, abs_vocab={ABS_VOCAB}, total={model.total_vocab_size.item()}")
print(f"Device: {device}")

## 1. Load a sample batch from GSM8K

In [ ]:
ds = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)
print(f"Dataset: {len(ds)} samples")

# Grab a small batch
B = 2
batch = {k: torch.stack([ds[i][k] for i in range(B)]).to(device)
         for k in ("input_ids", "attention_mask")}
batch["prompt_len"] = torch.tensor([ds[i]["prompt_len"] for i in range(B)], device=device)

ids  = batch["input_ids"]
attn = batch["attention_mask"]
pl   = batch["prompt_len"]

# Show raw text
for b in range(B):
    valid = attn[b].sum().item()
    text = tokenizer.decode(ids[b, :int(valid)], skip_special_tokens=False)
    print(f"\n--- Sample {b} (prompt_len={pl[b].item()}, valid={int(valid)}) ---")
    print(text[:500])

## 2. Build compressed sequence: `[query][abs×K][answer]`

Remove CoT tokens, insert K abstract placeholders between query and answer.

In [ ]:
def build_compressed_seq(ids, attn, pl, n_abs, bv, pad_id, answer_token_id=820):
    """Standalone version of SoRLTrainerv8._build_compressed_seq for inspection."""
    B, L = ids.shape
    ans_start = get_answer_start_index(ids, answer_token_id=answer_token_id)
    valid_len = attn.sum(dim=1)
    ans_len = (valid_len - ans_start).clamp(min=0)
    max_comp_len = int((pl + n_abs + ans_len).max().item())

    comp_data = ids.new_full((B, max_comp_len), pad_id)
    comp_attn = attn.new_zeros(B, max_comp_len)
    placeholder = bv  # first abstract token id

    for b in range(B):
        p, a, al = pl[b].item(), ans_start[b].item(), int(ans_len[b].item())
        comp_data[b, :p] = ids[b, :p]
        comp_attn[b, :p] = 1
        comp_data[b, p:p + n_abs] = placeholder
        comp_attn[b, p:p + n_abs] = 1
        if al > 0:
            comp_data[b, p + n_abs:p + n_abs + al] = ids[b, a:a + al]
            comp_attn[b, p + n_abs:p + n_abs + al] = 1

    ans_pos_s = pl + n_abs
    return comp_data, comp_attn, ans_start, ans_pos_s

# Build compressed
comp_data, comp_attn, ans_pos_t, ans_pos_s = build_compressed_seq(
    ids, attn, pl, K, bv, tokenizer.pad_token_id, ANSWER_TOK)

# Visualize original vs compressed for each sample
for b in range(B):
    p = pl[b].item()
    valid_orig = int(attn[b].sum().item())
    valid_comp = int(comp_attn[b].sum().item())
    a_t = ans_pos_t[b].item()
    a_s = ans_pos_s[b].item()

    print(f"\n{'='*70}")
    print(f"Sample {b}  |  prompt_len={p}  |  answer_tok @ orig[{a_t}] → comp[{a_s}]")
    print(f"Original length: {valid_orig}  →  Compressed length: {valid_comp}")
    print(f"CoT tokens removed: {valid_orig - valid_comp + K}")

    # Decode segments of compressed
    query_toks = comp_data[b, :p]
    abs_toks   = comp_data[b, p:p+K]
    ans_toks   = comp_data[b, p+K:valid_comp]

    print(f"\n  [QUERY] ({p} toks): {tokenizer.decode(query_toks)[:120]}...")
    print(f"  [ABS×{K}]: token_ids = {abs_toks.tolist()}")
    print(f"  [ANSWER] ({valid_comp - p - K} toks): {tokenizer.decode(ans_toks)[:200]}")

    # Show what was removed (CoT)
    cot_toks = ids[b, p:a_t]
    print(f"  [REMOVED CoT] ({a_t - p} toks): {tokenizer.decode(cot_toks)[:200]}...")

## 3. Teacher forward: `L_base` (trained!) + `h_teacher` at `####` position

In v8, `L_base` is part of the training loss (with grad). `h_teacher` is **detached** for KD.

In [ ]:
# Teacher forward — in training this runs WITH grad; here we use no_grad for inspection
with torch.no_grad():
    lab = ids.clone(); lab[attn == 0] = -100
    si = torch.arange(lab.size(1), device=device).unsqueeze(0)
    lab[si < pl.unsqueeze(1)] = -100

    teacher_out = model(
        input_ids=ids, attention_mask=attn,
        output_hidden_states=True)

    lg = teacher_out.logits.clone()
    lg[:, :, bv:] = -float("inf")
    base_loss = nn.CrossEntropyLoss(ignore_index=-100)(
        lg[:, :-1].contiguous().view(-1, lg.size(-1)),
        lab[:, 1:].contiguous().view(-1))

    # h_teacher detached — stop-gradient target for KD
    h_teacher = teacher_out.hidden_states[-1]  # (B, L, D)
    n_layers = len(teacher_out.hidden_states) - 1
    del teacher_out, lg

# Gather h_teacher at #### position
t_idx = ans_pos_t.clamp(max=h_teacher.size(1) - 1)
h_t = h_teacher[torch.arange(B, device=device), t_idx]  # (B, D)

print(f"L_base (teacher CE, TRAINED in v8): {base_loss.item():.4f}")
print(f"h_teacher shape: {h_teacher.shape}  (B, L, D)")
print(f"h_t shape (at #### pos): {h_t.shape}  (B, D)")
print(f"Number of transformer layers: {n_layers}")
for b in range(B):
    print(f"  Sample {b}: #### at pos {t_idx[b].item()}, "
          f"h_t norm = {h_t[b].norm().item():.2f}, "
          f"h_t mean = {h_t[b].mean().item():.4f}")

## 4. Student forward: recursion iterations on compressed sequence

At each iteration:
- Forward compressed `[query][abs×K][answer]` through the wrapper
- Sample new abstract tokens from logits
- Compute `L_compress` (CE on answer NL tokens) and `L_KD` (L1 at `####`)
- Combined loss: **`L_base + α_traj · L_compress + α_kd · L_KD`**

In [ ]:
from sorl.selfroute import SoRLTrainerv6

N_ITER = 4

# Recursion mask: positions with abstract tokens
vocab_size_0 = model.vocab_sizes[0].to(device)
recursion_mask = (comp_data >= vocab_size_0)
recursion_mask[:, 0] = False

print(f"Abstract positions per sample:")
for b in range(B):
    abs_pos = recursion_mask[b].nonzero(as_tuple=True)[0].tolist()
    print(f"  Sample {b}: {abs_pos}")

# Run N_ITER recursion iterations, collect losses + hidden states
idx = comp_data.clone()
iter_results = []

with torch.no_grad():
    for it in range(N_ITER):
        student_out = model(
            input_ids=idx, attention_mask=comp_attn,
            output_hidden_states=True)
        logits = student_out.logits

        # Sample new abstract tokens
        idx_new = model.extract_and_sample(
            logits, idx.clone(), recursion_mask, temperature=0.0)

        # L_compress: CE on answer (NL) tokens
        compress_loss = SoRLTrainerv6._traj_loss_from_logits(
            logits, idx_new, comp_attn, pl, bv)

        # h_student at #### position
        h_student_all = student_out.hidden_states[-1]
        s_idx = ans_pos_s.clamp(max=h_student_all.size(1) - 1)
        h_s = h_student_all[torch.arange(B, device=device), s_idx]

        # L_KD
        kd_loss = F.l1_loss(h_s, h_t.detach())

        # Cosine similarity between teacher & student at #### pos
        cos_sim = F.cosine_similarity(h_s, h_t, dim=-1)

        # Collect abs token ids chosen this iteration
        abs_ids = [idx_new[b][recursion_mask[b]].tolist() for b in range(B)]

        iter_results.append({
            "base_loss": base_loss.item(),
            "compress_loss": compress_loss.item(),
            "kd_loss": kd_loss.item(),
            "cos_sim": cos_sim.tolist(),
            "h_s_norm": [h_s[b].norm().item() for b in range(B)],
            "abs_ids": abs_ids,
        })

        idx = idx_new.detach()
        del student_out, logits, h_student_all

# Print iteration-by-iteration summary
print(f"\n{'Iter':>4} | {'L_base':>8} | {'L_compress':>10} | {'L_KD':>8} | {'cos_sim[0]':>10} | {'cos_sim[1]':>10} | {'‖h_s‖[0]':>8} | {'‖h_s‖[1]':>8}")
print("-" * 90)
for i, r in enumerate(iter_results):
    print(f"{i:4d} | {r['base_loss']:8.4f} | {r['compress_loss']:10.4f} | {r['kd_loss']:8.4f} | "
          f"{r['cos_sim'][0]:10.4f} | {r['cos_sim'][1]:10.4f} | "
          f"{r['h_s_norm'][0]:8.2f} | {r['h_s_norm'][1]:8.2f}")

print(f"\n‖h_t‖ (teacher): {[h_t[b].norm().item() for b in range(B)]}")

## 5. Visualize: Abstract tokens chosen + KD alignment over iterations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) L_compress and L_KD over iterations
iters = list(range(N_ITER))
ax = axes[0]
ax.plot(iters, [r["compress_loss"] for r in iter_results], "o-", label="L_compress")
ax.plot(iters, [r["kd_loss"] for r in iter_results], "s-", label="L_KD")
ax.set_xlabel("Recursion iteration")
ax.set_ylabel("Loss")
ax.set_title("Losses over iterations")
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Cosine similarity between h_teacher and h_student at #### position
ax = axes[1]
for b in range(B):
    ax.plot(iters, [r["cos_sim"][b] for r in iter_results], "o-", label=f"Sample {b}")
ax.set_xlabel("Recursion iteration")
ax.set_ylabel("cos(h_teacher, h_student)")
ax.set_title("KD alignment at #### position")
ax.legend()
ax.grid(True, alpha=0.3)

# (c) Abstract token diversity: unique abs tokens per sample per iteration
ax = axes[2]
for b in range(B):
    n_unique = [len(set(r["abs_ids"][b])) for r in iter_results]
    ax.plot(iters, n_unique, "o-", label=f"Sample {b}")
ax.set_xlabel("Recursion iteration")
ax.set_ylabel("# unique abs tokens")
ax.set_title(f"Abstract token diversity (out of {K})")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Detailed view: hidden-state heatmap at `####` position

Compare teacher vs student hidden states (first 64 dims) across iterations.

In [ ]:
# Re-run iterations collecting full hidden vectors for heatmap
idx = comp_data.clone()
h_s_per_iter = []  # list of (B, D) tensors

with torch.no_grad():
    for it in range(N_ITER):
        out = model(input_ids=idx, attention_mask=comp_attn,
                    output_hidden_states=True)
        hs = out.hidden_states[-1]
        s_idx = ans_pos_s.clamp(max=hs.size(1) - 1)
        h_s_per_iter.append(hs[torch.arange(B, device=device), s_idx].cpu())
        idx = model.extract_and_sample(out.logits, idx.clone(), recursion_mask, temperature=0.0).detach()
        del out

h_t_cpu = h_t.cpu()
D_SHOW = 64  # show first 64 dims

fig, axes = plt.subplots(B, 1, figsize=(14, 3 * B))
if B == 1:
    axes = [axes]

for b in range(B):
    ax = axes[b]
    # Stack: row 0 = teacher, rows 1..N = student iterations
    mat = torch.stack([h_t_cpu[b, :D_SHOW]] + [h[b, :D_SHOW] for h in h_s_per_iter])
    im = ax.imshow(mat.numpy(), aspect="auto", cmap="RdBu_r",
                   vmin=-mat.abs().max().item(), vmax=mat.abs().max().item())
    ax.set_yticks(range(N_ITER + 1))
    ax.set_yticklabels(["teacher"] + [f"iter {i}" for i in range(N_ITER)])
    ax.set_xlabel(f"Hidden dim (first {D_SHOW})")
    ax.set_title(f"Sample {b}: hidden state at #### position")
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## 7. Sequence layout comparison

Token-level view: color-code each position by type (query / CoT / abs / answer / pad).

In [ ]:
import matplotlib.patches as mpatches

def token_type_map(toks, attn_mask, prompt_len, ans_start, bv):
    """0=pad, 1=query, 2=cot, 3=abs, 4=answer."""
    L = toks.size(0)
    types = torch.zeros(L, dtype=torch.long)
    for i in range(L):
        if attn_mask[i] == 0:
            types[i] = 0  # pad
        elif i < prompt_len:
            types[i] = 1  # query
        elif toks[i] >= bv:
            types[i] = 3  # abstract
        elif i >= ans_start:
            types[i] = 4  # answer
        else:
            types[i] = 2  # cot
    return types

cmap_colors = ["#eeeeee", "#4c72b0", "#dd8452", "#c44e52", "#55a868"]
labels = ["pad", "query", "cot", "abs", "answer"]

fig, axes = plt.subplots(B * 2, 1, figsize=(16, 1.5 * B * 2), gridspec_kw={"hspace": 0.6})

for b in range(B):
    # Original
    v_orig = int(attn[b].sum().item())
    types_orig = token_type_map(ids[b], attn[b], pl[b].item(), ans_pos_t[b].item(), bv)
    ax = axes[b * 2]
    colors_orig = [cmap_colors[t] for t in types_orig[:v_orig].tolist()]
    ax.barh(0, width=[1]*v_orig, left=range(v_orig), color=colors_orig, height=0.8, edgecolor="none")
    ax.set_xlim(0, max(v_orig, int(comp_attn[b].sum().item())) + 2)
    ax.set_yticks([])
    ax.set_title(f"Sample {b} — Original [{v_orig} tokens]", fontsize=10)
    ax.axvline(pl[b].item(), color="black", ls="--", lw=0.8, label="prompt_end")
    ax.axvline(ans_pos_t[b].item(), color="green", ls="--", lw=0.8, label="####")

    # Compressed
    v_comp = int(comp_attn[b].sum().item())
    types_comp = token_type_map(comp_data[b], comp_attn[b], pl[b].item(), ans_pos_s[b].item(), bv)
    ax = axes[b * 2 + 1]
    colors_comp = [cmap_colors[t] for t in types_comp[:v_comp].tolist()]
    ax.barh(0, width=[1]*v_comp, left=range(v_comp), color=colors_comp, height=0.8, edgecolor="none")
    ax.set_xlim(0, max(v_orig, v_comp) + 2)
    ax.set_yticks([])
    ax.set_title(f"Sample {b} — Compressed [{v_comp} tokens]", fontsize=10)
    ax.axvline(pl[b].item(), color="black", ls="--", lw=0.8)
    ax.axvline(ans_pos_s[b].item(), color="green", ls="--", lw=0.8)

patches = [mpatches.Patch(color=c, label=l) for c, l in zip(cmap_colors, labels)]
fig.legend(handles=patches, loc="lower center", ncol=5, fontsize=9)
plt.suptitle("v8 Pipeline: Original vs Compressed sequence layout", fontsize=12, y=1.02)
plt.show()

## 8. Summary: total loss computation

Combined loss per iteration: **`L_base + α_traj · L_compress + α_kd · L_KD`**

`L_base` is constant across iterations (computed once from teacher); it anchors the full-CoT path while KD pulls the compressed representation toward it.

In [ ]:
alpha_traj = 1.0
alpha_kd   = 1.0

print(f"{'Iter':>4} | {'L_base':>8} | {'L_compress':>10} | {'L_KD':>8} | {'Total':>8} | abs tokens (sample 0)")
print("-" * 90)
for i, r in enumerate(iter_results):
    total = r["base_loss"] + alpha_traj * r["compress_loss"] + alpha_kd * r["kd_loss"]
    abs_str = str([tok - bv for tok in r["abs_ids"][0]])  # offset by bv for readability
    print(f"{i:4d} | {r['base_loss']:8.4f} | {r['compress_loss']:10.4f} | {r['kd_loss']:8.4f} | {total:8.4f} | {abs_str}")

print(f"\nInterpretation:")
print(f"  - L_base:     CE on full [query][cot][answer] — keeps the model grounded (TRAINED)")
print(f"  - L_compress: CE on [query][abs][answer] — student learns to predict answers from abs tokens")
print(f"  - L_KD:       L1 at #### position — pulls student repr toward teacher's (sg on teacher)")
print(f"  - Total:      L_base + α_traj·L_compress + α_kd·L_KD")
print(f"  - Without L_base in the loss, L_KD has no useful target to drag toward")

# TODO: Delete this cell (duplicate of cell 0)